# Lab 07 extra: How `async` and `await` work

**Going further, for Lab 07**

## Goal

Every call to the Agents SDK is written `await Runner.run(...)`. Understand that `await` from the
beginning: what an `async` function returns, what `await` does with it, and why two model calls that wait at the
same time finish in about the time of the slower one.

## What you will learn

- What a coroutine is, and how `await` runs it.
- Why two waits take the sum of their times one after the other, and the time of the slower one with
  `asyncio.gather`.
- Why a blocking call such as `time.sleep` inside an `async` function still blocks.
- How to make two real model calls at once with `AsyncOpenAI`.

## Setup

The same first cell as every lab: it reads the model name from `models.env` and takes the key from `.env` without
printing it.


In [1]:
from pathlib import Path
import asyncio
import json
import os
import re
import sqlite3
import time
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


## Wait for two model calls at once with async

A framework's `await` copied without understanding is where confusing errors start, so we go in small steps: a normal function, the same function with `async`, what `await` does, two waits one after the
other, two waits at once, then two real model calls.

### A normal function returns its value

A normal `def` runs from its first line to its `return`, and the caller gets the result.

In [2]:
def greet(name):
    return "hello " + name


print(greet("Helena"))
print("type of the return:", type(greet("Helena")).__name__)

hello Helena
type of the return: str


A string, as expected. Now the same body with `async` before `def`, which makes it a **coroutine function**. We call
it the same way and look at what comes back, without `await`.

In [3]:
async def greet_async(name):
    return "hello " + name


ticket = greet_async("Helena")
print(ticket)
print("type of the return:", type(ticket).__name__)

<coroutine object greet_async at 0x106a9c700>
type of the return: coroutine


The call did not run the body. It returned a **coroutine**: an object that holds work which has not started yet,
like a ticket for it. `"hello Helena"` exists nowhere so far.

`await` runs the coroutine to its end and gives back its result. In a notebook we can write `await` directly in a
cell, because Jupyter runs an **event loop** for us: the part of Python's `asyncio` that runs coroutines and switches
between them while they wait. In a `.py` file there is no running loop, so the top call goes inside
`asyncio.run(main())`.

In [4]:
result = await ticket
print(result)
print("type of the return:", type(result).__name__)

hello Helena
type of the return: str


Now the string. A coroutine runs once: awaiting `ticket` a second time would raise an error, so each call makes a new
one.

### Why `async` at all: a wait that lets go

`await` is not only a way to get a result. While a coroutine waits for something slow (a network reply, a database,
a timer), it can hand control back to the event loop, which runs other coroutines in the meantime. A model call is
almost all waiting: the request goes out, and our program has nothing to do until the reply comes back.

`asyncio.sleep(seconds)` is the simplest wait that lets go. `pause` prints when it starts and when it is done, so we
can see the order.

In [5]:
async def pause(name, seconds):
    print("start", name)
    await asyncio.sleep(seconds)
    print("done ", name)
    return name


print("returned:", await pause("alone", 0.3))

start alone


done  alone
returned: alone


### Two waits, one after the other

`await` one, then `await` the other: the second does not start until the first is done, so two one-second waits
should take about **two** seconds. `time.perf_counter()` reads a clock in seconds; the difference between two readings
is the time between them.

In [6]:
t0 = time.perf_counter()
first = await pause("a", 1)
second = await pause("b", 1)
print("results:", first, second)
print("one after the other:", round(time.perf_counter() - t0, 2), "seconds")

start a


done  a
start b


done  b
results: a b
one after the other: 2.0 seconds


`start a`, `done a`, then `start b`, `done b`: 2.0 seconds in all. This is how the hand-written loop in
notebook 1 ran its tool calls: one after another.

### Two waits at once with `asyncio.gather`

`asyncio.gather(...)` takes coroutines (the tickets, not their results), starts them all, and waits until the last
one is done. It returns a list of results **in the order of its arguments**, whichever finished first. Before you
run it: how long will two one-second waits take, and in which order will the four lines print?

In [7]:
t0 = time.perf_counter()
together = await asyncio.gather(pause("a", 1), pause("b", 1))
print("results:", together)
print("gathered:", round(time.perf_counter() - t0, 2), "seconds")

start a
start b


done  a
done  b
results: ['a', 'b']
gathered: 1.01 seconds


Both `start` lines come before both `done` lines: `b` started while `a` was still waiting. The clock says
1.01 seconds, the time of the slower wait rather than the sum, and the results come back as `['a', 'b']`,
in argument order.

**⚠️ A wait that does not let go blocks everything.** `time.sleep` is an ordinary wait: it holds the whole program,
event loop included, so nothing else runs during it. Put inside an `async def`, it still blocks, and `gather` cannot
overlap anything:

In [8]:
async def pause_blocking(name, seconds):
    print("start", name)
    time.sleep(seconds)  # an ordinary wait: the event loop cannot run anything else meanwhile
    print("done ", name)
    return name


t0 = time.perf_counter()
await asyncio.gather(pause_blocking("a", 1), pause_blocking("b", 1))
print("gathered, but blocking:", round(time.perf_counter() - t0, 2), "seconds")

start a


done  a
start b


done  b
gathered, but blocking: 2.01 seconds


2.01 seconds, and the lines print in sequence again: `async def` alone did nothing. The same thing happens with
any library call that waits without `await`, such as the ordinary `OpenAI` client: wrapped in an
`async def`, its calls still run one after another. For model calls at once we need a client whose calls are
coroutines.

### Two real model calls, one after the other and together

`AsyncOpenAI` is the same client as `OpenAI` with every method made a coroutine function, so each call must be
awaited. `ask` sends one question on the Responses API (the API the Agents SDK uses) and returns the reply's text:

- `input` is the question; `max_output_tokens` caps the reply;
- `reasoning={"effort": EFFORT or omit}` passes the effort from `models.env`; `omit` is the `openai` package's marker
  for "leave this out", so a model with no reasoning setting (an empty `REASONING_EFFORT`) gets none;
- `response.output_text` joins the text parts of the reply.

The two questions do not depend on each other, which is what makes running them together safe.

In [9]:
from openai import AsyncOpenAI, omit

aclient = AsyncOpenAI()


async def ask(question):
    response = await aclient.responses.create(
        model=MODEL,
        input=question,
        reasoning={"effort": EFFORT or omit},
        max_output_tokens=200,
    )
    return response.output_text


QUESTIONS = [
    "In one sentence, what is an invoice?",
    "In one sentence, what does a customer support rep do?",
]

t0 = time.perf_counter()
for question in QUESTIONS:
    print(await ask(question))
print("one after the other:", round(time.perf_counter() - t0, 2), "seconds")

An invoice is a document requesting payment for goods or services provided.


A customer support representative helps customers resolve issues, answer questions, and use a company’s products or services.
one after the other: 6.93 seconds


In [10]:
t0 = time.perf_counter()
answers = await asyncio.gather(*[ask(question) for question in QUESTIONS])
for answer in answers:
    print(answer)
print("gathered:", round(time.perf_counter() - t0, 2), "seconds")

An invoice is a document that lists the goods or services provided, their costs, and the amount due for payment.
A customer support representative helps customers resolve questions and problems with a company’s products or services.
gathered: 1.95 seconds


`[ask(question) for question in QUESTIONS]` builds a list of two coroutines without running them, and the `*` passes
them to `gather` as two separate arguments. Reading the two runs:

- one after the other: 6.93 seconds, the time of both calls added up;
- gathered: 1.95 seconds: the two waits overlapped, so the total is set by the slower call, not by the sum;
- the answers come back in the order of `QUESTIONS` either way.

The same call does not take the same time twice (a model call can vary by a second or more), so your seconds will
differ: compare the two totals, not the digits. Gathering is right only when the calls are independent: the desk agent in notebook 2
cannot gather `find_customer` with `get_invoices`, because the second needs the `customer_id` the first returns.

**In production:** `asyncio.TaskGroup` (Python 3.11 and later) does what `gather` does, and cancels the other tasks
when one fails; a server that answers many people at once is built on the same event loop.

**Checkpoint.** Two one-second sleeps take about 2 seconds awaited one after the other and about 1 second gathered,
with results in argument order; `time.sleep` inside `async def` stays at about 2; two real calls gathered take about
as long as the slower one. The Agents SDK does this for us when a model asks for two tools in one reply.

## What you can do now

- Say what an `async def` function returns when it is called without `await`.
- Run two independent waits at once with `asyncio.gather`, and say in which order the results come back.
- Spot a blocking call inside an `async` function.
- Make two model calls at once with `AsyncOpenAI`.

## Resources

- [Python `asyncio`: coroutines and tasks](https://docs.python.org/3/library/asyncio-task.html): `await`, `gather`,
  `TaskGroup`
- [OpenAI Python library: async usage](https://github.com/openai/openai-python#async-usage): `AsyncOpenAI`

Back to [notebook 1](07a_meet_the_agents_sdk.ipynb) or [notebook 2](07b_agents_sdk_at_the_desk.ipynb).
